In [1]:
!pip install nltk rouge-score
!pip install bert-score

In [2]:
!wget https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip .
!unzip BLEURT-20.zip    

--2025-04-14 15:51:15--  https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 2a00:1450:4009:827::201b, 2a00:1450:4009:81f::201b, 2a00:1450:4009:822::201b, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|2a00:1450:4009:827::201b|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2140294207 (2.0G) [application/octet-stream]
Saving to: ‘BLEURT-20.zip’

BLEURT-20.zip       100%[===================>]   1.99G  17.3MB/s    in 2m 1s   

2025-04-14 15:53:17 (16.8 MB/s) - ‘BLEURT-20.zip’ saved [2140294207/2140294207]

--2025-04-14 15:53:17--  http://./
Resolving . (.)... failed: No address associated with hostname.
wget: unable to resolve host address ‘.’
FINISHED --2025-04-14 15:53:17--
Total wall clock time: 2m 2s
Downloaded: 1 files, 2.0G in 2m 1s (16.8 MB/s)
Archive:  BLEURT-20.zip
   creating: BLEURT-20/
  inflating: BLEURT-20/bert_config.json  
  inflating: BLEURT-20/saved_model.pb 

In [3]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download('wordnet')
nltk.download('punkt')

def compute_bleu(reference, candidate):
    """
    Compute BLEU score for BLEU-1, BLEU-2, BLEU-3, and BLEU-4 between reference and candidate.
    Uses a smoothing function for short sentences.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    smoothie = SmoothingFunction().method1  # Smoothing for short sequences
    
    bleu_scores = {}
    for n in range(1, 5):
        weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
        bleu_scores[f"BLEU-{n}"] = sentence_bleu(reference_tokens, candidate_tokens, weights=weights, smoothing_function=smoothie)
    
    return bleu_scores

def compute_rouge(reference, candidate):
    """
    Compute ROUGE-L, ROUGE-1, and ROUGE-2 scores.
    Returns the F1 scores.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return {k: v.fmeasure for k, v in scores.items()}

def compute_meteor(reference, candidate):
    """
    Compute METEOR score.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    return meteor_score(reference_tokens, candidate_tokens)

import bert_score.score
def compute_bertscore(reference, candidate, lang='en', model_type='bert-base-uncased'):
    P, R, F1 = bert_score.score([candidate], [reference], lang=lang, model_type=model_type, verbose=False)

    return {
        'precision': P[0].item(),
        'recall': R[0].item(),
        'f1': F1[0].item()
    }

import bleurt.score
def compute_bleurt(references, candidates, checkpoint_path="BLEURT-20"):
    scorer = bleurt.score.BleurtScorer(checkpoint_path)
    scores = scorer.score(references=references, candidates=candidates)
    return scores

[nltk_data] Downloading package wordnet to /home/gp921526/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/gp921526/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/home/gp921526/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-14 15:53:31.664275: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-14 15:53:31.673386: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been reg

In [ ]:
import pandas as pd
import os

results_folder = "results"
result_filename = "results/ablation/pororo_ablation_language_gpt_4o_mini.csv"
evaluation_results_folder = "results/evaluation"

os.makedirs(evaluation_results_folder, exist_ok=True)

result = pd.read_csv(result_filename)

os.makedirs(evaluation_results_folder, exist_ok=True)

for i, row in result.iterrows():

    if i == len(result)-1: # avoid the last row with total and average values
        continue

    reference_answer = row["correct_answer"].lower()
    generated_answer = row["predicted_answer"].lower()

    bleus = compute_bleu(reference_answer, generated_answer)
    rouges = compute_rouge(reference_answer, generated_answer)
    meteor = compute_meteor(reference_answer, generated_answer)
    berts = compute_bertscore(reference_answer, generated_answer)
    
    for n in range(1, 5):
        result.at[i, f"BLEU-{n}"] = bleus[f"BLEU-{n}"]
    
    for rouge_type, score_value in rouges.items():
        result.at[i, f"{rouge_type.upper()}"] = score_value
    
    result.at[i, "METEOR"] = meteor

    result.at[i, "BERTScore_Precision"] = berts["precision"]
    result.at[i, "BERTScore_Recall"] = berts["recall"]
    result.at[i, "BERTScore_F1"] = berts["f1"]


reference_answers = list(result["correct_answer"])[:-1]
generated_answers = list(result["predicted_answer"])[:-1]
bleurts = compute_bleurt(reference_answers, generated_answers)
result["BLEURT"] = bleurts + [""]

output_filename = os.path.basename(result_filename)

result.to_csv(os.path.join(evaluation_results_folder, output_filename), index=False)


In [ ]:
result